## Particionando Dataset Eliteserien

In [1]:
import os
import shutil
import random
from pathlib import Path
import zipfile

In [2]:
nome_zip = "Eliteserien.zip"

with zipfile.ZipFile(nome_zip, 'r') as zip_ref:
    zip_ref.extractall()

O dataset possui 750 amostras balanceadas em 3 anos (ou seja, 250 amostras por ano)

Vamos selecionar aleatoriamente de modo que tenhamos 3 pastas no dataset particionado: treino (525), teste (131) e validação (95), de forma que fique 175 por ano (treinamento), 25 (validacao) e 50 (teste). Vamos contar pela pasta frames.

In [3]:
caminho_dataset = Path("Eliteserien")
caminho_destino = Path("Eliteserien_Particionado")

In [5]:
anos = ["2021", "2022", "2023"]
ids_especificos = []

for ano in caminho_dataset.iterdir():
  aux = []
  if ano.is_dir():
    caminho_pasta = Path(ano/"frames")
    for id_geral in caminho_pasta.iterdir():
      for id_especifico in id_geral.iterdir():
        aux.append(id_especifico.stem)
    ids_especificos.append(aux)

print(ids_especificos[0][0])


2961_0236


In [6]:
quantidade_treino = 175
quantidade_validacao = 25
quantidade_teste = 50

extensoes = {
    "detection": ".txt",
    "frames": ".jpg",
    "segmentation": ".txt"
}

for i, ano in enumerate(anos):
    random.shuffle(ids_especificos[i])

    splits = {
        "train": ids_especificos[i][:quantidade_treino],
        "val": ids_especificos[i][quantidade_treino:quantidade_treino+quantidade_validacao],
        "test": ids_especificos[i][quantidade_treino+quantidade_validacao:]
    }

    for split_name, id_list in splits.items():
        for item_id in id_list:

            id_geral = item_id.split("_")[0]

            for categoria in ["detection", "frames", "segmentation"]:

                extensao = extensoes[categoria]

                src_file = (
                    caminho_dataset /
                    ano /
                    categoria /
                    id_geral /
                    f"{item_id}{extensao}"
                )

                if src_file.exists():

                    dst_folder = (
                        caminho_destino /
                        split_name /
                        ano /
                        categoria /
                        id_geral
                    )

                    os.makedirs(dst_folder, exist_ok=True)

                    shutil.copy2(
                        src_file,
                        dst_folder / src_file.name
                    )

                else:
                    print(f"Arquivo não encontrado: {src_file}")

    print(f"Ano {ano} processado.")

Ano 2021 processado.
Ano 2022 processado.
Ano 2023 processado.


In [9]:
for split in ["train", "val", "test"]:
    pasta_segmentation = caminho_destino / split

    quantidade = sum(
        1
        for arquivo in pasta_segmentation.rglob("*.jpg")
    )

    print(f"{split}: {quantidade} imagens")

train: 525 imagens
val: 75 imagens
test: 150 imagens


In [10]:
originais = sum(
    1 for arq in (caminho_dataset).rglob("*.jpg")
)

particionados = sum(
    1 for arq in (caminho_destino).rglob("*.jpg")
)

print("Originais:", originais)
print("Particionados:", particionados)
print("OK?" , originais == particionados)

Originais: 750
Particionados: 750
OK? True
